In [2]:
import ee
import geemap
import ipywidgets as widgets
from ipyleaflet import WidgetControl, GeoJSON
import math
import json
import pandas as pd
import time
import osmnx as ox
from shapely.geometry import shape, mapping, box

# ============================================================
# 1. INICIO DE GOOGLE EARTH ENGINE
# ============================================================
PROYECTO = 'booming-list-511608-s2'
try:
    ee.Initialize(project=PROYECTO)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROYECTO)


# ============================================================
# 2. MAPA (con arreglo del error 'east ... None')
# ============================================================
class MapaSeguro(geemap.Map):
    """Mapa que ignora los limites vacios (None) que manda el navegador."""
    def set_state(self, sync_data):
        claves_limites = ('north', 'south', 'east', 'west', 'bounds')
        limpio = {k: v for k, v in sync_data.items()
                  if not (k in claves_limites and v is None)}
        super().set_state(limpio)


Map = MapaSeguro(center=[22.58, -102.52], zoom=13, height='680px')

# ============================================================
# 3. IMAGEN SENTINEL-2
# ============================================================
sentinel_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                       .filterDate('2026-01-01', '2026-10-01')
                       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15)))
imagen_base = sentinel_collection.median().divide(10000)

# ============================================================
# 4. WIDGETS (controles de la interfaz)
# ============================================================
estilo = {'description_width': 'initial'}

index_selector = widgets.Dropdown(
    options=[
        ('NDVI (Indice de Vegetacion Normalizado)', 'NDVI'),
        ('NDWI (Indice de Agua Normalizado)', 'NDWI'),
        ('MNDWI (Indice de Agua Modificado)', 'MNDWI'),
        ('EVI (Indice de Vegetacion Mejorado)', 'EVI'),
        ('NDSI (Indice de Nieve Normalizado)', 'NDSI'),
        ('BSI (Indice de Suelo Desnudo)', 'BSI'),
        ('SAVI (Indice de Vegetacion Ajustado al Suelo)', 'SAVI'),
        ('NDMI (Indice de Humedad Normalizado)', 'NDMI'),
    ],
    value='MNDWI', description='Estudio:', style=estilo,
    layout=widgets.Layout(width='300px'))

input_radio = widgets.FloatText(
    value=50.0, description='Distancia(m):', style=estilo,
    layout=widgets.Layout(width='130px'))

tipo_forma_marker = widgets.Dropdown(
    options=[('Circulo', 'circulo'), ('Cuadrado', 'cuadrado')],
    value='cuadrado', description='Marcador:', style=estilo,
    layout=widgets.Layout(width='125px'))

figura_selector = widgets.Dropdown(
    options=[('Sin figuras', -1)], value=-1,
    description='Zona Activa:', style=estilo,
    layout=widgets.Layout(width='320px'))

filtro_osm = widgets.Dropdown(
    options=[('Todo', 'todo'), ('Edificios', 'edificio'), ('Calles', 'calle'),
             ('Agua', 'agua'), ('Servicios', 'servicio'),
             ('Naturaleza / uso de suelo', 'natural'), ('Otras estructuras', 'estructura')],
    value='todo', description='Mostrar:', style=estilo,
    layout=widgets.Layout(width='220px'))

btn_procesar = widgets.Button(description='Procesar Zona', button_style='success',
                            layout=widgets.Layout(width='120px'))
btn_exportar = widgets.Button(description='Exportar Pixeles', button_style='info',
                            layout=widgets.Layout(width='125px'))
btn_cargar_osm = widgets.Button(description='Cargar OSM', button_style='warning',
                                layout=widgets.Layout(width='110px'))
btn_limpiar = widgets.Button(description='Limpiar Todo', button_style='danger',
                           layout=widgets.Layout(width='115px'))

# Variables globales de estado
control_leyenda_actual = None
control_indicador_actual = None
historial_geometrias = []
capas_agregadas = []
dibujos_registrados = 0
capa_osm = None
osm_features = []
osm_geoms = {}
silenciar_centrado = False

# LÍMITES AMPLIADOS AL MÁXIMO PERMITIDO POR SERVIDORES PÚBLICOS DE OSM
MAX_KM2 = 25        # Área máxima ampliada a 25 km²
MAX_DIBUJO = 15000  # Límite de renderizado visual ampliado a 15,000 estructuras


# ============================================================
# 5. INDICES ESPECTRALES
# ============================================================
def obtener_imagen_indice(estudio):
    b = imagen_base.select
    if estudio == 'NDVI':
        img = imagen_base.normalizedDifference(['B8', 'B4']).rename('NDVI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#8c510a', '#dfc27d', '#f5f5f5', '#80cdc1', '#01665e']}
        etiquetas = ['Muy Densa (0.8)', 'Densa (0.6)', 'Moderada (0.4)', 'Baja (0.2)', 'Suelo/Escaso (0.0)']
    elif estudio == 'NDWI':
        img = imagen_base.normalizedDifference(['B3', 'B8']).rename('NDWI')
        vis = {'min': -0.3, 'max': 0.5, 'palette': ['#d73027', '#fee08b', '#ffffbf', '#91bfdb', '#4575b4']}
        etiquetas = ['Agua Alta (0.5)', 'Agua Media (0.3)', 'Humedad (0.0)', 'Transicion (-0.1)', 'Suelo Seco (-0.3)']
    elif estudio == 'MNDWI':
        img = imagen_base.normalizedDifference(['B3', 'B11']).rename('MNDWI')
        vis = {'min': -0.5, 'max': 0.4, 'palette': ['#a50026', '#fdae61', '#ffffbf', '#abd9e9', '#08519c']}
        etiquetas = ['Agua Libre (0.4)', 'Agua Somera (0.2)', 'Transicion (0.0)', 'Suelo (-0.2)', 'Urbano / Roca (-0.5)']
    elif estudio == 'EVI':
        img = imagen_base.expression(
            '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
            {'NIR': b('B8'), 'RED': b('B4'), 'BLUE': b('B2')}).rename('EVI')
        vis = {'min': 0.0, 'max': 1.0, 'palette': ['#f7fcb9', '#addd8e', '#78c679', '#31a354', '#006837']}
        etiquetas = ['Muy Alta (1.0)', 'Alta (0.75)', 'Media (0.5)', 'Baja (0.25)', 'Minima (0.0)']
    elif estudio == 'NDSI':
        img = imagen_base.normalizedDifference(['B3', 'B11']).rename('NDSI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#000000', '#c6dbef', '#9ecae1', '#4292c6', '#ffffff']}
        etiquetas = ['Nieve Alta (0.8)', 'Media (0.6)', 'Baja (0.4)', 'Minima (0.2)', 'Sin Nieve (0.0)']
    elif estudio == 'BSI':
        img = imagen_base.expression(
            '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
            {'SWIR': b('B11'), 'RED': b('B4'), 'NIR': b('B8'), 'BLUE': b('B2')}).rename('BSI')
        vis = {'min': -0.4, 'max': 0.4, 'palette': ['#3288bd', '#abdda4', '#ffffbf', '#fdae61', '#d53e4f']}
        etiquetas = ['Suelo Denso (0.4)', 'Alto (0.2)', 'Neutro (0.0)', 'Bajo (-0.2)', 'Vegetacion/Agua (-0.4)']
    elif estudio == 'SAVI':
        img = imagen_base.expression(
            '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
            {'NIR': b('B8'), 'RED': b('B4')}).rename('SAVI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#ffffcc', '#a1dab4', '#41b6c4', '#225ea8', '#0c2c84']}
        etiquetas = ['Alta (0.8)', 'Media-Alta (0.6)', 'Media (0.4)', 'Baja (0.2)', 'Minima (0.0)']
    else:  # NDMI
        img = imagen_base.normalizedDifference(['B8', 'B11']).rename('NDMI')
        vis = {'min': -0.4, 'max': 0.4, 'palette': ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#4575b4']}
        etiquetas = ['Alta Humedad (0.4)', 'Media-Alta (0.2)', 'Moderada (0.0)', 'Baja (-0.2)', 'Muy Seco (-0.4)']
    return img, vis, estudio, etiquetas


# ============================================================
# 6. FUNCIONES AUXILIARES DE INTERFAZ
# ============================================================
def refrescar_selector(seleccionar_ultima=True):
    if historial_geometrias:
        figura_selector.options = [(item['nombre'], i) for i, item in enumerate(historial_geometrias)]
        if seleccionar_ultima:
            figura_selector.value = len(historial_geometrias) - 1
    else:
        figura_selector.options = [('Sin figuras', -1)]
        figura_selector.value = -1


def nombre_unico(base):
    usados = {item['nombre'] for item in historial_geometrias}
    if base not in usados:
        return base
    n = 2
    while f"{base} ({n})" in usados:
        n += 1
    return f"{base} ({n})"


def actualizar_indicador_activo(nombre_zona):
    global control_indicador_actual
    if control_indicador_actual is not None:
        try:
            Map.remove_control(control_indicador_actual)
        except Exception:
            pass

    html = f"""
    <div style="background: rgba(255,255,255,0.95); padding: 8px 15px; border-radius: 8px;
                box-shadow: 0 0 15px rgba(0,0,0,0.3); font-family: sans-serif; font-size: 12px;
                font-weight: bold; color: #007bff; text-align: center;">
        {nombre_zona}
    </div>"""
    control_indicador_actual = WidgetControl(widget=widgets.HTML(value=html), position='topright')
    Map.add_control(control_indicador_actual)


def mostrar_leyenda(titulo, vis, etiquetas):
    global control_leyenda_actual
    if control_leyenda_actual is not None:
        try:
            Map.remove_control(control_leyenda_actual)
        except Exception:
            pass

    p = vis['palette']
    html = f"""
    <div style="background: white; padding: 10px; border-radius: 5px; box-shadow: 0 0 15px rgba(0,0,0,0.2);
                font-family: sans-serif; font-size: 10px; width: 230px;">
        <b>{titulo}</b><br>
        <div style="background: linear-gradient(to top, {p[0]}, {p[1]}, {p[2]}, {p[3]}, {p[4]});
                    width: 18px; height: 120px; float: left; margin-right: 8px; margin-top: 5px;"></div>
        <div style="height: 120px; display: inline-block; line-height: 24px;">
            {'<br>'.join(etiquetas)}
        </div>
    </div>"""
    control_leyenda_actual = WidgetControl(widget=widgets.HTML(value=html), position='bottomleft')
    Map.add_control(control_leyenda_actual)


# ============================================================
# 7. NOMBRES Y CATEGORIAS DE LAS ESTRUCTURAS DE OPENSTREETMAP
# ============================================================
TIPOS_AMENITY = {
    'hospital': 'Hospital', 'clinic': 'Clinica', 'doctors': 'Consultorio',
    'pharmacy': 'Farmacia', 'school': 'Escuela', 'university': 'Universidad',
    'college': 'Colegio', 'kindergarten': 'Kinder', 'library': 'Biblioteca',
    'restaurant': 'Restaurante', 'cafe': 'Cafeteria', 'bank': 'Banco',
    'fuel': 'Gasolinera', 'parking': 'Estacionamiento', 'police': 'Policia',
    'fire_station': 'Bomberos', 'townhall': 'Presidencia municipal',
    'place_of_worship': 'Iglesia/Templo', 'marketplace': 'Mercado',
    'cinema': 'Cine', 'theatre': 'Teatro', 'bus_station': 'Central de autobuses',
}
TIPOS_NATURAL = {
    'water': 'Cuerpo de agua', 'wetland': 'Humedal', 'wood': 'Bosque',
    'scrub': 'Matorral', 'grassland': 'Pastizal', 'bare_rock': 'Roca expuesta',
    'sand': 'Arena', 'beach': 'Playa', 'peak': 'Cerro', 'cliff': 'Acantilado',
}
TIPOS_WATER = {
    'lake': 'Lago', 'pond': 'Estanque', 'reservoir': 'Presa/Embalse',
    'river': 'Rio', 'canal': 'Canal', 'basin': 'Cuenca', 'stream_pool': 'Poza',
}
TIPOS_WATERWAY = {
    'river': 'Rio', 'stream': 'Arroyo', 'canal': 'Canal',
    'drain': 'Drenaje', 'ditch': 'Zanja', 'dam': 'Cortina de presa',
}
TIPOS_LANDUSE = {
    'residential': 'Zona residencial', 'commercial': 'Zona comercial',
    'industrial': 'Zona industrial', 'farmland': 'Terreno agricola',
    'farmyard': 'Granja', 'forest': 'Bosque', 'grass': 'Area de pasto',
    'meadow': 'Pradera', 'orchard': 'Huerto', 'cemetery': 'Panteon',
    'retail': 'Zona de comercios', 'quarry': 'Cantera', 'reservoir': 'Presa/Embalse',
    'military': 'Zona militar', 'recreation_ground': 'Area recreativa',
}
TIPOS_LEISURE = {
    'park': 'Parque', 'garden': 'Jardin', 'pitch': 'Cancha',
    'stadium': 'Estadio', 'playground': 'Juegos infantiles',
    'sports_centre': 'Centro deportivo', 'golf_course': 'Campo de golf',
}
TIPOS_BUILDING = {
    'house': 'Casa', 'apartments': 'Departamentos', 'commercial': 'Edificio comercial',
    'industrial': 'Nave industrial', 'church': 'Iglesia', 'school': 'Escuela',
    'hospital': 'Hospital', 'warehouse': 'Bodega', 'retail': 'Comercio',
}
TIPOS_MANMADE = {
    'bridge': 'Puente', 'tower': 'Torre', 'water_tower': 'Torre de agua',
    'storage_tank': 'Tanque', 'pipeline': 'Tuberia', 'wastewater_plant': 'Planta de aguas residuales',
    'water_works': 'Planta de agua', 'chimney': 'Chimenea',
}
TIPOS_RAILWAY = {'rail': 'Via de tren', 'station': 'Estacion de tren', 'platform': 'Anden'}

ESTILOS = {
    'edificio':    {'color': '#8b5a2b', 'fillColor': '#c68642', 'weight': 1, 'fillOpacity': 0.6},
    'calle':       {'color': '#f28e1c', 'fillColor': '#f28e1c', 'weight': 3, 'fillOpacity': 0.3},
    'agua':        {'color': '#1f78b4', 'fillColor': '#5fa8e8', 'weight': 2, 'fillOpacity': 0.6},
    'servicio':    {'color': '#c0392b', 'fillColor': '#e74c3c', 'weight': 2, 'fillOpacity': 0.55},
    'natural':     {'color': '#3c8d40', 'fillColor': '#7fc97f', 'weight': 1, 'fillOpacity': 0.4},
    'estructura': {'color': '#555555', 'fillColor': '#999999', 'weight': 2, 'fillOpacity': 0.5},
}


def valor(fila, columna):
    v = fila.get(columna)
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    return str(v)


def describir_osm(fila):
    nombre = valor(fila, 'name')
    for col, tabla, defecto in [
        ('amenity', TIPOS_AMENITY, 'Servicio'),
        ('highway', {}, 'Calle'),
        ('waterway', TIPOS_WATERWAY, 'Cauce'),
        ('water', TIPOS_WATER, 'Cuerpo de agua'),
        ('natural', TIPOS_NATURAL, 'Zona natural'),
        ('building', TIPOS_BUILDING, 'Edificio'),
        ('man_made', TIPOS_MANMADE, 'Estructura'),
        ('railway', TIPOS_RAILWAY, 'Via de tren'),
        ('leisure', TIPOS_LEISURE, 'Area recreativa'),
        ('landuse', TIPOS_LANDUSE, 'Uso de suelo'),
    ]:
        v = valor(fila, col)
        if v is None:
            continue
        if col == 'building' and v == 'yes':
            return 'Edificio', nombre
        return tabla.get(v, defecto), nombre
    return 'Zona', nombre


def categoria_osm(fila):
    if valor(fila, 'amenity'):
        return 'servicio'
    if valor(fila, 'highway'):
        return 'calle'
    if valor(fila, 'waterway') or valor(fila, 'water') or valor(fila, 'natural') in ('water', 'wetland'):
        return 'agua'
    if valor(fila, 'building'):
        return 'edificio'
    if valor(fila, 'man_made') or valor(fila, 'railway'):
        return 'estructura'
    return 'natural'


def geometria_a_poligono(geom, grande=False):
    if geom.geom_type in ('LineString', 'MultiLineString'):
        return geom.buffer(0.00015)
    if geom.geom_type in ('Point', 'MultiPoint'):
        return geom.buffer(0.0003 if grande else 0.0001)
    return geom


# ============================================================
# 8. REGISTRAR LOS DIBUJOS DEL MAPA EN LA LISTA
# ============================================================
def dibujo_a_geometria(dibujo):
    if hasattr(dibujo, 'geometry'):
        return dibujo.geometry()
    if isinstance(dibujo, dict):
        return ee.Geometry(dibujo.get('geometry', dibujo))
    return ee.Feature(dibujo).geometry()


def registrar_dibujos_nuevos():
    global dibujos_registrados
    if not hasattr(Map, 'draw_features'):
        return

    distancia_m = input_radio.value
    forma = tipo_forma_marker.value

    for n in range(dibujos_registrados, len(Map.draw_features)):
        try:
            g = dibujo_a_geometria(Map.draw_features[n])
            tipo_g = g.type().getInfo()

            if tipo_g == 'Point':
                lon, lat = g.coordinates().getInfo()[:2]
                if forma == 'cuadrado':
                    d_lat = distancia_m / 111320.0
                    d_lon = distancia_m / (111320.0 * math.cos(math.radians(lat)))
                    geom = ee.Geometry.BBox(lon - d_lon, lat - d_lat, lon + d_lon, lat + d_lat)
                    nombre = f"Dibujo {n + 1}: Cuadrado de {int(distancia_m * 2)} m"
                else:
                    geom = g.buffer(distancia_m)
                    nombre = f"Dibujo {n + 1}: Circulo de {int(distancia_m)} m de radio"
            else:
                geom = g
                nombres_tipo = {'Polygon': 'Poligono', 'LineString': 'Linea', 'Rectangle': 'Rectangulo'}
                nombre = f"Dibujo {n + 1}: {nombres_tipo.get(tipo_g, tipo_g)}"

            historial_geometrias.append({'geometry': geom, 'nombre': nombre_unico(nombre),
                                         'origen': 'dibujo', 'osm_id': None})
        except Exception as e:
            print(f"No se pudo leer el dibujo {n + 1}: {e}")

    if len(Map.draw_features) > dibujos_registrados:
        dibujos_registrados = len(Map.draw_features)
        refrescar_selector()


# ============================================================
# 9. ESTRUCTURAS DE OPENSTREETMAP DIBUJADAS EN EL MAPA 
# ============================================================
ox.settings.timeout = 300  # Aumentamos el tiempo límite a 5 minutos para que traiga edificios sin que saque fallos
ox.settings.overpass_endpoint = "https://overpass-api.de/api/interpreter"

def obtener_area_estudio():
    registrar_dibujos_nuevos()

    dibujos = [it for it in historial_geometrias if it['origen'] == 'dibujo']
    i = figura_selector.value
    if 0 <= i < len(historial_geometrias) and historial_geometrias[i]['origen'] == 'dibujo':
        elegido = historial_geometrias[i]
    elif dibujos:
        elegido = dibujos[-1]
    else:
        elegido = None

    if elegido is not None:
        poligono = geometria_a_poligono(shape(elegido['geometry'].getInfo()))
        return poligono, elegido['nombre']

    limites = Map.bounds
    if limites and len(limites) == 2:
        (s, w), (n, e) = limites
        return box(w, s, e, n), 'area visible del mapa'
    return None, None


def estimar_km2(poligono):
    minx, miny, maxx, maxy = poligono.bounds
    lat = (miny + maxy) / 2
    ancho = (maxx - minx) * 111.32 * math.cos(math.radians(lat))
    alto = (maxy - miny) * 111.32
    return ancho * alto


def estilo_estructura(feature):
    return ESTILOS.get(feature['properties']['categoria'], ESTILOS['estructura'])


def al_clic_estructura(event=None, feature=None, **kwargs):
    global silenciar_centrado
    if feature is None:
        return
    props = feature['properties']
    osm_id = props['id']

    silenciar_centrado = True
    try:
        for idx, item in enumerate(historial_geometrias):
            if item['osm_id'] == osm_id:
                figura_selector.value = idx
                actualizar_indicador_activo(item['nombre'])
                return

        geom = geometria_a_poligono(osm_geoms[osm_id], grande=True)
        historial_geometrias.append({
            'geometry': ee.Geometry(mapping(geom)),
            'nombre': nombre_unico(props['etiqueta']),
            'origen': 'osm',
            'osm_id': osm_id,
        })
        refrescar_selector()
        actualizar_indicador_activo(historial_geometrias[-1]['nombre'])
    finally:
        silenciar_centrado = False


def dibujar_capa_osm(*args):
    global capa_osm
    if capa_osm is not None:
        try:
            Map.remove_layer(capa_osm)
        except Exception:
            pass
        capa_osm = None

    filtro = filtro_osm.value
    seleccion = [f for f in osm_features
                 if filtro == 'todo' or f['properties']['categoria'] == filtro]
    if not seleccion:
        return
    if len(seleccion) > MAX_DIBUJO:
        print(f"Hay {len(seleccion)} estructuras; se dibujan las primeras {MAX_DIBUJO}. "
              "Usa el filtro 'Mostrar' para ver el resto.")
        seleccion = seleccion[:MAX_DIBUJO]

    coleccion = {'type': 'FeatureCollection', 'features': seleccion}
    capa_osm = GeoJSON(
        data=coleccion,
        style_callback=estilo_estructura,
        hover_style={'weight': 4, 'color': '#ffff00', 'fillOpacity': 0.8},
        name='Estructuras OSM')
    capa_osm.on_click(al_clic_estructura)
    Map.add_layer(capa_osm)


filtro_osm.observe(dibujar_capa_osm, names='value')


def cargar_datos_osm(b):
    global osm_features, osm_geoms
    print("Buscando estructuras (edificios, calles, servicios) en OpenStreetMap...")
    try:
        poligono, origen = obtener_area_estudio()
        if poligono is None:
            print("Dibuja una figura (o mueve el mapa) antes de cargar OSM.")
            return

        km2 = estimar_km2(poligono)
        if km2 > MAX_KM2:
            print(f"El area es de {km2:.1f} km2 (maximo permitido: {MAX_KM2} km2). Dibuja una zona mas pequena.")
            return

        print(f"Area: {origen} (~{km2:.2f} km2). Descargando, puede tardar unos segundos...")
        
        # Tags detallados para asegurar la captura de edificios y puntos de interés
        tags_interes = {
            'amenity': True, 'highway': True, 'waterway': True, 'water': True,
            'natural': True, 'landuse': True, 'leisure': True, 'building': True,
            'man_made': True, 'railway': True,
        }

        intentos = 4
        espera = 5
        gdf = None
        
        endpoints = [
            "https://overpass-api.de/api/interpreter",
            "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter"
        ]

        for intento in range(intentos):
            try:
                ox.settings.overpass_endpoint = endpoints[intento % len(endpoints)]
                gdf = ox.features_from_polygon(poligono, tags=tags_interes)
                if not gdf.empty:
                    break
            except Exception as ex:
                print(f"Intento {intento+1} fallido: {ex}")
                if intento < intentos - 1:
                    print(f"Esperando {espera} segundos y cambiando de espejo Overpass...")
                    time.sleep(espera)
                    espera *= 2
                else:
                    print("Error: Los servidores de OpenStreetMap están saturados o la zona es muy densa. Inténtalo con un área un poco menor.")
                    return

        if gdf is None or gdf.empty:
            print("No se encontraron estructuras en esta area.")
            return

        osm_features = []
        osm_geoms = {}
        for n, (_, fila) in enumerate(gdf.iterrows()):
            geom = fila.geometry
            if geom is None or geom.is_empty:
                continue

            tipo, nombre = describir_osm(fila)
            etiqueta = f"{tipo}: {nombre}" if nombre else f"{tipo} (sin nombre)"
            osm_geoms[n] = geom

            osm_features.append({
                'type': 'Feature',
                'geometry': json.loads(json.dumps(mapping(geometria_a_poligono(geom)))),
                'properties': {'id': n, 'tipo': tipo, 'nombre': nombre or '',
                               'categoria': categoria_osm(fila), 'etiqueta': etiqueta},
            })

        dibujar_capa_osm()

        resumen = {}
        for f in osm_features:
            c = f['properties']['categoria']
            resumen[c] = resumen.get(c, 0) + 1
        detalle = ', '.join(f"{k}: {v}" for k, v in sorted(resumen.items()))
        print(f"Listo: {len(osm_features)} estructuras cargadas ({detalle}).")
        print("Haz clic en una para seleccionarla y luego presiona 'Procesar Zona'.")
    except Exception as ex:
        print(f"Error en la consulta a OpenStreetMap: {ex}")


btn_cargar_osm.on_click(cargar_datos_osm)


# ============================================================
# 10. PROCESAR LA ZONA SELECCIONADA
# ============================================================
def ejecutar_analisis_individual(b):
    estudio = index_selector.value
    registrar_dibujos_nuevos()

    i = figura_selector.value
    if not (0 <= i < len(historial_geometrias)):
        print("Atencion: dibuja una figura o haz clic en una estructura de OSM antes de procesar.")
        return

    item = historial_geometrias[i]
    geometria = item['geometry']
    nombre_zona = item['nombre']

    layer_img, vis, titulo, etiquetas = obtener_imagen_indice(estudio)
    nombre_capa = f"{nombre_zona} - {estudio}"

    Map.addLayer(layer_img.clip(geometria), vis, nombre_capa)
    capas_agregadas.append(Map.layers[-1])

    mostrar_leyenda(titulo, vis, etiquetas)
    actualizar_indicador_activo(nombre_zona)
    try:
        Map.center_object(geometrio if 'geometrio' in locals() else geometria)
    except Exception:
        pass

    print(f"Proceso completado para '{nombre_zona}' con el indice: {titulo}.")


btn_procesar.on_click(ejecutar_analisis_individual)


# ============================================================
# 11. CAMBIAR DE ZONA ACTIVA
# ============================================================
def al_cambiar_figura(change):
    i = change['new']
    if isinstance(i, int) and 0 <= i < len(historial_geometrias):
        item = historial_geometrias[i]
        actualizar_indicador_activo(item['nombre'])
        if silenciar_centrado:
            return
        try:
            Map.center_object(item['geometry'])
        except Exception:
            pass


figura_selector.observe(al_cambiar_figura, names='value')


# ============================================================
# 12. PANEL DE CONTROLES
# ============================================================
panel_controles = widgets.VBox([
    widgets.HBox([index_selector, figura_selector, filtro_osm]),
    widgets.HBox([input_radio, tipo_forma_marker, btn_procesar, btn_exportar, btn_cargar_osm, btn_limpiar])
], layout=widgets.Layout(min_width='880px', min_height='90px', padding='4px'))

control_menu = WidgetControl(widget=panel_controles, position='bottomright')
Map.add_control(control_menu)


# ============================================================
# 13. LIMPIAR TODO
# ============================================================
def limpiar_todo_sistema(b):
    global control_leyenda_actual, control_indicador_actual, dibujos_registrados
    global capa_osm, osm_features, osm_geoms

    historial_geometrias.clear()
    refrescar_selector()

    for capa in capas_agregadas:
        try:
            Map.remove_layer(capa)
        except Exception:
            pass
    capas_agregadas.clear()

    if capa_osm is not None:
        try:
            Map.remove_layer(capa_osm)
        except Exception:
            pass
    capa_osm = None
    osm_features = []
    osm_geoms = {}

    try:
        if hasattr(Map, 'draw_control') and Map.draw_control is not None:
            Map.draw_control.clear()
    except Exception:
        pass
        
    if hasattr(Map, 'draw_features'):
        Map.draw_features.clear()
        
    dibujos_registrados = 0

    for ctrl in (control_leyenda_actual, control_indicador_actual):
        if ctrl is not None:
            try:
                Map.remove_control(ctrl)
            except Exception:
                pass
    control_leyenda_actual = None
    control_indicador_actual = None

    print("Sistema limpiado por completo.")


btn_limpiar.on_click(limpiar_todo_sistema)


# ============================================================
# 14. EXPORTAR PIXELES DE LA ZONA ACTIVA A GOOGLE DRIVE
# ============================================================
import unicodedata
import re

def exportar_pixeles_zona(b):
    registrar_dibujos_nuevos()

    i = figura_selector.value
    if not (0 <= i < len(historial_geometrias)):
        print("Atencion: primero elige o dibuja una zona.")
        return

    item = historial_geometrias[i]
    estudio = index_selector.value
    layer_img, _, _, _ = obtener_imagen_indice(estudio)

    # Limpieza estricta de acentos y caracteres especiales para el nombre del archivo
    nombre_crudo = item['nombre']
    nombre_sin_tildes = ''.join(c for c in unicodedata.normalize('NFD', nombre_crudo) if unicodedata.category(c) != 'Mn')
    nombre_limpio = re.sub(r'[^a-zA-Z0-9._,:;\-]', '_', nombre_sin_tildes)[:60]

    print(f"Enviando exportacion de '{item['nombre']}' a Google Drive...")
    try:
        # 1. Creamos la muestra con la imagen, lon y lat
        muestra = (layer_img.select(estudio)
                   .addBands(ee.Image.pixelLonLat())
                   .sample(region=item['geometry'], scale=10))
        muestra_limpia = muestra.select([estudio, 'longitude', 'latitude'])

        descripcion_tarea = f'export_{estudio}_{nombre_limpio}'[:100]
        prefijo_archivo = f'pixeles_{estudio}_{nombre_limpio}'

        tarea = ee.batch.Export.table.toDrive(
            collection=muestra_limpia,
            description=descripcion_tarea,
            fileFormat='CSV',
            fileNamePrefix=prefijo_archivo)
        tarea.start()
        print("Exportacion iniciada sin la columna .geo. ¡Revisa tu Google Drive!")
    except Exception as ex:
        print(f"Error en la exportacion: {ex}")


btn_exportar.on_click(exportar_pixeles_zona)

# Mostrar el mapa
Map

MapaSeguro(center=[22.58, -102.52], controls=(WidgetControl(options=['position', 'transparent_bg'], position='…